In [1]:
import os
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

print("✓ pandas:", pd.__version__)
print("✓ numpy:", np.__version__)
print("✓ yfinance:", yf.__version__)
print("✓ All imports successful")

✓ pandas: 3.0.6
✓ numpy: 2.5.3
✓ yfinance: 1.7.0
✓ All imports successful


In [2]:
PAGE_SIZE = 250

REGIONS = [
    "us",
    "in",
    "ca",
    "gb",
    "de",
    "fr",
    "jp",
    "hk",
    "au",
    "sg",
    "br",
    "cn",
]

In [3]:
# =========================================================
# GET EQUITIES
# =========================================================


def get_equities(region, max_results=10000):

    print(f"\n{'=' * 60}")
    print(f"Downloading {region.upper()} equities")
    print(f"{'=' * 60}")

    all_rows = []
    offset = 0

    while offset < max_results:
        try:
            query = yf.EquityQuery("eq", ["region", region])
            result = yf.screen(
                query, offset=offset, size=PAGE_SIZE, sortField="ticker", sortAsc=True
            )
            quotes = result.get("quotes", [])
            if not quotes:
                print("No more results.")
                break

            # Store EVERYTHING Yahoo returned
            for item in quotes:
                item = dict(item)
                # Add our own metadata
                item["_region"] = region
                item["_asset_source"] = "Yahoo Finance"
                item["_downloaded_at"] = datetime.now(timezone.utc).isoformat()

                all_rows.append(item)

            print(
                f"{region.upper()} | "
                f"Offset: {offset:,} | "
                f"Downloaded: {len(quotes):,} | "
                f"Total: {len(all_rows):,}"
            )

            if len(quotes) < PAGE_SIZE:
                break

            offset += PAGE_SIZE
            time.sleep(1)

        except Exception as e:  # noqa: BLE001

            print(f"ERROR | Region={region} " f"Offset={offset}: {e}")

            break

    return all_rows

In [4]:
# =========================================================
# GET ETFs
# =========================================================
def get_etfs(region="us", max_results=10000):
    print(f"\n{'=' * 60}")
    print(f"Downloading {region.upper()} ETFs")
    print(f"{'=' * 60}")
    all_rows = []
    offset = 0
    while offset < max_results:
        try:
            query = yf.ETFQuery("eq", ["region", region])
            result = yf.screen(
                query, offset=offset, size=PAGE_SIZE, sortField="ticker", sortAsc=True
            )
            quotes = result.get("quotes", [])
            if not quotes:
                break
            for item in quotes:
                item = dict(item)
                item["_region"] = region
                item["_asset_source"] = "Yahoo Finance"
                item["_asset_type"] = "ETF"
                item["_downloaded_at"] = datetime.now(timezone.utc).isoformat()
                all_rows.append(item)
            print(
                f"{region.upper()} ETF | "
                f"Offset: {offset:,} | "
                f"Downloaded: {len(quotes):,} | "
                f"Total: {len(all_rows):,}"
            )
            if len(quotes) < PAGE_SIZE:
                break
            offset += PAGE_SIZE
            time.sleep(1)
        except Exception as e:  # noqa: BLE001
            print(f"ETF ERROR | " f"Offset={offset}: {e}")
            break
    return all_rows

In [5]:
# =========================================================
# MAIN
# =========================================================

def main():
    all_data = []
    # -----------------------------------------------------
    # Equities
    # -----------------------------------------------------
    for region in REGIONS:
        rows = get_equities(region=region, max_results=10000)
        all_data.extend(rows)

    # -----------------------------------------------------
    # ETFs
    # -----------------------------------------------------

    for region in REGIONS:
        rows = get_etfs(region=region, max_results=10000)
        all_data.extend(rows)

    # -----------------------------------------------------
    # Remove duplicate symbols
    # -----------------------------------------------------
    unique = {}
    for item in all_data:
        symbol = item.get("symbol")
        if symbol and symbol not in unique:
            unique[symbol] = item
    all_data = list(unique.values())

    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    print("\n")
    print("=" * 70)
    print("FINAL DATABASE")
    print("=" * 70)

    print(f"Total unique tickers: " f"{len(all_data):,}")

    df = pd.DataFrame(all_data)

    if df.empty:
        print("No data to save.")
        return
    # -----------------------------------------------------
    # Ensure ticker column exists
    # -----------------------------------------------------
    if "symbol" in df.columns:
        df = df.rename(columns={"symbol": "ticker"})
    # -----------------------------------------------------
    # Remove duplicate tickers
    # -----------------------------------------------------
    if "ticker" in df.columns:
        df = df.drop_duplicates(subset=["ticker"], keep="first")
        df = df.sort_values("ticker")
        df = df.dropna(subset=["shortName"])

    print("\nDONE.")
    return df

In [6]:
df = main()


US | Offset: 0 | Downloaded: 250 | Total: 250
US | Offset: 250 | Downloaded: 250 | Total: 500
US | Offset: 500 | Downloaded: 250 | Total: 750
US | Offset: 750 | Downloaded: 250 | Total: 1,000
US | Offset: 1,000 | Downloaded: 250 | Total: 1,250
US | Offset: 1,250 | Downloaded: 250 | Total: 1,500
US | Offset: 1,500 | Downloaded: 250 | Total: 1,750
US | Offset: 1,750 | Downloaded: 250 | Total: 2,000
US | Offset: 2,000 | Downloaded: 250 | Total: 2,250
US | Offset: 2,250 | Downloaded: 250 | Total: 2,500
US | Offset: 2,500 | Downloaded: 250 | Total: 2,750
US | Offset: 2,750 | Downloaded: 250 | Total: 3,000
US | Offset: 3,000 | Downloaded: 250 | Total: 3,250
US | Offset: 3,250 | Downloaded: 250 | Total: 3,500
US | Offset: 3,500 | Downloaded: 250 | Total: 3,750
US | Offset: 3,750 | Downloaded: 250 | Total: 4,000
US | Offset: 4,000 | Downloaded: 250 | Total: 4,250
US | Offset: 4,250 | Downloaded: 250 | Total: 4,500
US | Offset: 4,500 | Downloaded: 250 | Total: 4,750
US | Offset: 4,750 | Downlo

In [7]:
df

,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,currency,regularMarketChangePercent,hasPrePostMarketData,...,tickerChangeDate,ytdReturn,prevExchange,exchangeTransferDate,delistingDate,underlyingSymbol,expireDate,expireIsoDate,openInterest,_asset_type
29098,en-US,US,EQUITY,Equity,Delayed Quote,False,LOW,EUR,-0.568181,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29099,en-US,US,EQUITY,Equity,Delayed Quote,False,LOW,EUR,-8.108107,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29100,en-US,US,EQUITY,Equity,Delayed Quote,False,LOW,EUR,-3.428568,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29101,en-US,US,EQUITY,Equity,Delayed Quote,False,LOW,EUR,-2.285712,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29102,en-US,US,EQUITY,Equity,Delayed Quote,False,LOW,EUR,0.000000,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81788,en-US,US,ETF,ETF,Free Realtime Quote,False,LOW,USD,0.513427,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ETF
81789,en-US,US,ETF,ETF,Free Realtime Quote,False,LOW,USD,0.563142,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ETF
81790,en-US,US,ETF,ETF,Free Realtime Quote,False,LOW,USD,1.645345,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ETF
81791,en-US,US,ETF,ETF,Delayed Quote,False,LOW,USD,0.131248,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ETF


In [8]:
# Save
df.to_csv("yahoo_finance_tickers.csv", index=False, encoding="utf-8-sig")
print(f"\nCSV saved:" f"\n{os.path.abspath('yahoo_finance_tickers.csv')}")

print(f"\nTotal tickers: {len(df):,}")

print(f"Total fields: {len(df.columns):,}")

print("\nFields stored:")
for column in df.columns:
    print(f"  - {column}")


CSV saved:
c:\lovely ram\Work\pappus\stocks_recomend\yahoo_finance_tickers.csv

Total tickers: 87,285
Total fields: 108

Fields stored:
  - language
  - region
  - quoteType
  - typeDisp
  - quoteSourceName
  - triggerable
  - customPriceAlertConfidence
  - currency
  - regularMarketChangePercent
  - hasPrePostMarketData
  - firstTradeDateMilliseconds
  - priceHint
  - regularMarketChange
  - regularMarketTime
  - regularMarketPrice
  - regularMarketDayHigh
  - regularMarketDayRange
  - regularMarketDayLow
  - regularMarketVolume
  - regularMarketPreviousClose
  - bidSize
  - askSize
  - market
  - messageBoardId
  - fullExchangeName
  - longName
  - regularMarketOpen
  - averageDailyVolume3Month
  - averageDailyVolume10Day
  - corporateActions
  - fiftyTwoWeekLowChange
  - fiftyTwoWeekLowChangePercent
  - fiftyTwoWeekRange
  - fiftyTwoWeekHighChange
  - fiftyTwoWeekHighChangePercent
  - fiftyTwoWeekChangePercent
  - dividendDate
  - earningsTimestamp
  - earningsTimestampStart
  - ea